# 🌲🌲🌲 Random Forest — Wine Quality Prediction

---

## 🎯 What Will You Learn?

- Understand **ensemble learning** (wisdom of the crowd)
- Learn how **Random Forest** combines many weak trees into a strong model
- Understand **bagging** and **feature randomness**
- Explore **feature importance** — which wine properties matter most?
- Tune hyperparameters for better performance

---

## 📖 Table of Contents
1. What is Random Forest?
2. Loading and Exploring the Wine Data
3. Data Preprocessing
4. Building the Random Forest Model
5. Feature Importance Analysis
6. Hyperparameter Tuning
7. Final Evaluation
8. Key Takeaways


---
## 1. 🌲 What is Random Forest?

### The "Expert Committee" Analogy 🏛️

Imagine you need a medical diagnosis. What's better:
- **Option A:** Ask ONE doctor → risky if that doctor is wrong
- **Option B:** Ask 100 different doctors and take the majority vote → much more reliable!

**Random Forest = 100 different doctors (decision trees), each voting!**

### Two Sources of Randomness

**Randomness 1: Bootstrap Sampling (Bagging)**
```
Original Data: [Patient1, Patient2, Patient3 ... Patient918]

Tree 1 gets:  [Patient5, Patient1, Patient5, Patient7, Patient2...] (random sample with replacement)
Tree 2 gets:  [Patient3, Patient9, Patient1, Patient2, Patient8...] (different random sample)
Tree 3 gets:  [Patient1, Patient1, Patient4, Patient6, Patient3...] (yet another sample)
...
```
Each tree sees a slightly different version of the data → each tree is different!

**Randomness 2: Feature Randomness**
```
All features: [Acidity, Sugar, Alcohol, pH, Sulfates...]

Tree 1 can only use: [Acidity, Sugar, Alcohol]  ← 3 random features
Tree 2 can only use: [pH, Sulfates, Density]    ← different 3 random features
Tree 3 can only use: [Alcohol, Chlorides, pH]   ← yet another 3
```
This prevents all trees from making the same splits!

### Why is This Better Than One Decision Tree?
- Single tree: can overfit, unstable (change one data point → very different tree)
- Random Forest: errors of individual trees cancel out → much more stable and accurate!

### The Voting Process
```
New wine sample → all 100 trees vote:
  85 trees say: "Good wine!" (quality ≥ 7)
  15 trees say: "Not good wine" (quality < 7)

Final answer: GOOD WINE (85/100 = 85% confidence)
```

In [ ]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries imported!")

---
## 2. 🍷 Loading and Exploring the Wine Data

In [ ]:
# Load the Wine Quality dataset
df = pd.read_csv('data/WineQT.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Drop the 'Id' column — it's just a row number, not a useful feature
df = df.drop('Id', axis=1)

print("Wine quality distribution:")
print(df['quality'].value_counts().sort_index())
print(f"\nQuality range: {df['quality'].min()} to {df['quality'].max()}")

### 🍷 Understanding the Wine Quality Data

Each row is a wine sample with chemical measurements:

| Feature | What it means | Why it matters |
|---------|--------------|----------------|
| `fixed acidity` | Non-volatile acids (tartaric acid) | Contributes to wine's crisp taste |
| `volatile acidity` | Acetic acid (vinegar-like) | Too much → bad taste |
| `citric acid` | Adds freshness | Small amounts add flavor |
| `residual sugar` | Sugar after fermentation | Sweetness level |
| `chlorides` | Salt content | Too much = salty taste |
| `free sulfur dioxide` | Free SO2 | Preservative |
| `total sulfur dioxide` | Total SO2 | Too much → sulfur smell |
| `density` | Mass per volume | Related to alcohol/sugar |
| `pH` | Acidity level | Scale of 0-14 |
| `sulphates` | Wine additive | Antimicrobial |
| `alcohol` | Alcohol percentage | Major quality factor |
| `quality` | **Score 0-10 by experts** | **Our target!** |

In [ ]:
# Convert to BINARY classification: Good (1) vs Not Good (0)
# A score of 7 or above = 'Good Quality Wine'
# A score below 7 = 'Not Good Quality'
df['good_wine'] = (df['quality'] >= 7).astype(int)

good_counts = df['good_wine'].value_counts()
print(f"Binary Wine Quality Distribution:")
print(f"  Not Good (score < 7):  {good_counts[0]} wines ({good_counts[0]/len(df)*100:.1f}%)")
print(f"  Good     (score >= 7): {good_counts[1]} wines ({good_counts[1]/len(df)*100:.1f}%)")

In [ ]:
# Visualize the wine data
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Quality score distribution
df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], 
    color='#9b59b6', edgecolor='black', width=0.6)
axes[0,0].axvline(x=3.5, color='red', linestyle='--', linewidth=2, label='Good threshold (7+)')
axes[0,0].set_title('Wine Quality Score Distribution', fontweight='bold')
axes[0,0].set_xlabel('Quality Score')
axes[0,0].set_ylabel('Count')
axes[0,0].legend()

# 2. Alcohol vs Quality
df.boxplot(column='alcohol', by='quality', ax=axes[0,1])
plt.sca(axes[0,1])
axes[0,1].set_title('Alcohol % by Quality Score', fontweight='bold')
axes[0,1].set_xlabel('Quality Score')
axes[0,1].set_ylabel('Alcohol (%)')
plt.suptitle('')

# 3. Correlation heatmap
corr = df.drop('good_wine', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=axes[0,2], annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, mask=mask, linewidths=0.3, annot_kws={'size': 7})
axes[0,2].set_title('Feature Correlation Matrix', fontweight='bold')

# 4. Volatile acidity vs binary quality
df[df['good_wine']==0]['volatile acidity'].hist(ax=axes[1,0], bins=25, alpha=0.7,
                                                  color='#e74c3c', label='Not Good')
df[df['good_wine']==1]['volatile acidity'].hist(ax=axes[1,0], bins=25, alpha=0.7,
                                                  color='#2ecc71', label='Good')
axes[1,0].set_title('Volatile Acidity: Good vs Not Good', fontweight='bold')
axes[1,0].set_xlabel('Volatile Acidity')
axes[1,0].legend()

# 5. Alcohol vs binary quality
df[df['good_wine']==0]['alcohol'].hist(ax=axes[1,1], bins=25, alpha=0.7,
                                        color='#e74c3c', label='Not Good')
df[df['good_wine']==1]['alcohol'].hist(ax=axes[1,1], bins=25, alpha=0.7,
                                        color='#2ecc71', label='Good')
axes[1,1].set_title('Alcohol: Good vs Not Good', fontweight='bold')
axes[1,1].set_xlabel('Alcohol (%)')
axes[1,1].legend()

# 6. Sulphates vs quality
df[df['good_wine']==0]['sulphates'].hist(ax=axes[1,2], bins=25, alpha=0.7,
                                          color='#e74c3c', label='Not Good')
df[df['good_wine']==1]['sulphates'].hist(ax=axes[1,2], bins=25, alpha=0.7,
                                          color='#2ecc71', label='Good')
axes[1,2].set_title('Sulphates: Good vs Not Good', fontweight='bold')
axes[1,2].set_xlabel('Sulphates')
axes[1,2].legend()

plt.tight_layout()
plt.savefig('wine_eda.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 3. 🔧 Data Preprocessing

In [ ]:
# Features and Target
X = df.drop(['quality', 'good_wine'], axis=1)  # Drop both original quality and our new label
y = df['good_wine']

print(f"Features shape: {X.shape}")
print(f"Feature names: {list(X.columns)}")
print(f"\nMissing values: {X.isnull().sum().sum()}")
print(f"Target distribution: {y.value_counts().to_dict()}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {len(X_train)} samples")
print(f"Testing:  {len(X_test)} samples")
print(f"\nNote: Random Forest does NOT require scaling (like Decision Trees)")
print(f"But we'll scale for fair comparison with SVM in the next notebook.")

---
## 4. 🌲 Building the Random Forest Model

In [ ]:
# First, let's compare: Single Decision Tree vs Random Forest
from sklearn.tree import DecisionTreeClassifier

# Single Decision Tree
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
tree_train_acc = accuracy_score(y_train, single_tree.predict(X_train))
tree_test_acc = accuracy_score(y_test, single_tree.predict(X_test))

print("Single Decision Tree:")
print(f"  Train Accuracy: {tree_train_acc*100:.2f}% (overfit!)")
print(f"  Test Accuracy:  {tree_test_acc*100:.2f}%")
print()

In [ ]:
# Random Forest with default settings
# n_estimators = number of trees (default: 100)
rf_model = RandomForestClassifier(
    n_estimators=100,   # 100 trees in the forest
    random_state=42,    # For reproducibility
    n_jobs=-1           # Use all CPU cores (faster training)
)

print("🚀 Training Random Forest with 100 trees...")
rf_model.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test))

print("\nRandom Forest (100 trees):")
print(f"  Train Accuracy: {rf_train_acc*100:.2f}%")
print(f"  Test Accuracy:  {rf_test_acc*100:.2f}%")
print()
print(f"📈 Random Forest improvement over single tree: +{(rf_test_acc - tree_test_acc)*100:.2f}%")

In [ ]:
# How does the number of trees affect performance?
tree_counts = [1, 5, 10, 25, 50, 100, 200, 500]
train_accs = []
test_accs = []

for n in tree_counts:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, rf.predict(X_train))*100)
    test_accs.append(accuracy_score(y_test, rf.predict(X_test))*100)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(tree_counts, train_accs, 'b-o', linewidth=2, markersize=6,
        label='Training Accuracy', color='#3498db')
ax.plot(tree_counts, test_accs, 'r-o', linewidth=2, markersize=6,
        label='Test Accuracy', color='#e74c3c')
ax.axhline(y=tree_test_acc*100, color='gray', linestyle=':', 
           label=f'Single Tree ({tree_test_acc*100:.1f}%)')
ax.set_xlabel('Number of Trees (n_estimators)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Effect of Number of Trees on Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('rf_n_estimators.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n💡 Observation: More trees = better accuracy, but diminishing returns after ~100")

---
## 5. 🔍 Feature Importance Analysis

One of the best features of Random Forest is that it tells you **which features matter most** for prediction.

In [ ]:
# Extract feature importances from the trained model
feature_names = X.columns
importances = rf_model.feature_importances_

# Get standard deviation of importances across all trees (shows stability)
std_importances = np.std([tree.feature_importances_ for tree in rf_model.estimators_], axis=0)

# Create a DataFrame for easy viewing
fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances,
    'Std': std_importances
}).sort_values('Importance', ascending=False)

print("Feature Importances (sorted by importance):")
print(fi_df.round(4).to_string(index=False))
print(f"\n✅ Top 3 most important features for wine quality prediction:")
for i, row in fi_df.head(3).iterrows():
    print(f"   {i+1 if False else fi_df.index.get_loc(i)+1}. {row['Feature']}: {row['Importance']:.4f} ({row['Importance']*100:.1f}%)")

In [ ]:
# Visualize Feature Importances with error bars
fig, ax = plt.subplots(figsize=(10, 7))

fi_sorted = fi_df.sort_values('Importance', ascending=True)
colors = plt.cm.RdYlGn(fi_sorted['Importance'] / fi_sorted['Importance'].max())

bars = ax.barh(fi_sorted['Feature'], fi_sorted['Importance'],
               xerr=fi_sorted['Std'], color=colors, edgecolor='black',
               capsize=4, error_kw={'linewidth': 1.5})
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=12)
ax.set_title('Random Forest Feature Importance\n(Wine Quality Prediction)', 
             fontsize=14, fontweight='bold')

for bar, val in zip(bars, fi_sorted['Importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n💡 Alcohol is the strongest predictor of wine quality!")

---
## 6. ⚙️ Hyperparameter Tuning

**Hyperparameters** are the settings we choose BEFORE training. Finding the best settings improves model performance.

Key Random Forest hyperparameters:
| Parameter | What it controls | Default |
|-----------|-----------------|--------|
| `n_estimators` | Number of trees | 100 |
| `max_depth` | Max tree depth | None (unlimited) |
| `max_features` | Features per tree | 'sqrt' (√n features) |
| `min_samples_split` | Min samples to split | 2 |
| `min_samples_leaf` | Min samples in leaf | 1 |

In [ ]:
# Simple hyperparameter exploration (faster than full GridSearch)
results = []

for n_est in [50, 100, 200]:
    for max_d in [5, 10, None]:
        for max_f in ['sqrt', 'log2']:
            rf = RandomForestClassifier(
                n_estimators=n_est, max_depth=max_d,
                max_features=max_f, random_state=42, n_jobs=-1
            )
            rf.fit(X_train, y_train)
            test_acc = accuracy_score(y_test, rf.predict(X_test))
            test_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
            results.append({
                'n_estimators': n_est, 'max_depth': str(max_d),
                'max_features': max_f, 'Test Accuracy': test_acc*100,
                'ROC-AUC': test_auc
            })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print("Top 5 hyperparameter combinations:")
print(results_df.head(5).round(4).to_string(index=False))

In [ ]:
# Build the best model from our search
best_params = results_df.iloc[0]
print(f"Best settings: n_estimators={best_params['n_estimators']}, "
      f"max_depth={best_params['max_depth']}, max_features={best_params['max_features']}")

best_rf = RandomForestClassifier(
    n_estimators=int(best_params['n_estimators']),
    max_depth=None if best_params['max_depth'] == 'None' else int(best_params['max_depth']),
    max_features=best_params['max_features'],
    random_state=42, n_jobs=-1
)
best_rf.fit(X_train, y_train)

final_acc = accuracy_score(y_test, best_rf.predict(X_test))
final_auc = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:,1])
print(f"\nFinal Model Performance:")
print(f"  Accuracy: {final_acc*100:.2f}%")
print(f"  ROC-AUC:  {final_auc:.4f}")

---
## 7. 📏 Final Evaluation

In [ ]:
# Detailed evaluation
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)

print("Final Random Forest — Classification Report:")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Not Good (<7)', 'Good (≥7)']))

In [ ]:
# Final visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['Pred: Not Good', 'Pred: Good'],
            yticklabels=['Actual: Not Good', 'Actual: Good'],
            linewidths=2, linecolor='white', annot_kws={'size': 18, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix\nRandom Forest', fontsize=13, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
axes[1].plot(fpr, tpr, lw=2.5, color='#27ae60', label=f'Random Forest (AUC={final_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#27ae60')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Random Forest', fontsize=13, fontweight='bold')
axes[1].legend()

# Single Tree vs RF comparison
models = ['Single\nDecision Tree', 'Random\nForest']
test_accs_compare = [tree_test_acc*100, final_acc*100]
bars = axes[2].bar(models, test_accs_compare, color=['#e74c3c', '#27ae60'], 
                   edgecolor='black', width=0.5)
axes[2].set_ylim([70, 100])
axes[2].set_ylabel('Test Accuracy (%)')
axes[2].set_title('Single Tree vs Random Forest', fontsize=13, fontweight='bold')
for bar, val in zip(bars, test_accs_compare):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('rf_final_evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-Validation for more reliable score estimate
cv_scores = cross_val_score(best_rf, X, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f"5-Fold Cross Validation Accuracy:")
for i, score in enumerate(cv_scores):
    print(f"  Fold {i+1}: {score*100:.2f}%")
print(f"  Mean: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
print()
print("💡 Cross-validation gives a more reliable estimate by testing on all data!")

---
## 8. 🎓 Key Takeaways

### Random Forest Summary:
- **Many decision trees + majority voting = Random Forest**
- Each tree sees a **different random sample** of data (bagging)
- Each tree uses a **random subset of features** (feature randomness)
- These two randomness sources make the forest very robust!

### Advantages of Random Forest:
| ✅ Advantage | Description |
|---|---|
| High Accuracy | Usually one of the best algorithms |
| Feature Importance | Tells you which features matter |
| Handles Missing Data | Can handle some missing values |
| No Scaling Needed | Works with raw numbers |
| Resistant to Overfitting | Many trees average out individual errors |

### Disadvantages:
| ⚠️ Disadvantage | Description |
|---|---|
| Slow for Prediction | Must query 100+ trees |
| Less Interpretable | Hard to explain the combined decision |
| Memory Intensive | Stores 100+ trees |

### Wine Quality Insights:
- **Alcohol** is the strongest predictor of quality
- **Volatile acidity** (too much vinegar taste) strongly predicts LOW quality
- **Sulphates** and **citric acid** contribute to good quality

### Next: SVM (Support Vector Machine)
A completely different approach — instead of many trees, it finds the **perfect dividing line** between classes!


In [ ]:
print("=" * 55)
print("         RANDOM FOREST SUMMARY")
print("=" * 55)
print(f"  Dataset:         Wine Quality")
print(f"  Total Wines:     {len(df)}")
print(f"  Target:          Good (≥7) vs Not Good (<7)")
print(f"  Single Tree Acc: {tree_test_acc*100:.2f}%")
print(f"  Random Forest:   {final_acc*100:.2f}%  ← Better!")
print(f"  ROC-AUC:         {final_auc:.4f}")
print(f"  Top Feature:     alcohol ({importances[list(feature_names).index('alcohol')]:.4f})")
print("=" * 55)